# Практикум 1. Основы анализа данных на финансовых рынках

**Курс «Анализ данных и машинное обучение на финансовых рынках»**

---

### Как работать с этим блокнотом

1. Выполняйте ячейки **по порядку**: `Shift + Enter`.
2. Подойдёт Jupyter (Anaconda или python.org + pip) или Google Colab — см. инструкцию в информационном
   разделе курса. Нужны numpy, pandas, scipy, matplotlib; sklearn, statsmodels и torch — по желанию (ячейки
   с ними необязательны и пропускаются, если библиотеки нет).
3. 📘 — разбор: запустите и посмотрите результат. ✏️ — задание: допишите код вместо `...` или `# ваш код`, затем запустите
   ячейку проверки с `assert` — если ошибок нет, решение верно.
   💡 — решение под треугольником, открывайте после честной попытки.
4. Перед сдачей: **Kernel → Restart & Run All**.

### Что будет к концу блокнота

Вы очистите «сырую» выгрузку котировок, учтёте дивиденды и сплит, посчитаете доходности, риск и
просадки, проверите стилизованные факты (толстые хвосты, кластеризацию волатильности), спрогнозируете VaR и проверите
прогноз бэктестом, оцените беты и сравните портфели на новых данных. Каждое задание проверяется ячейкой с `assert`.

In [ ]:
import json, os, sys, time, random
from datetime import datetime, timedelta, timezone
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 140)
BLUE, ORANGE = "#34519E", "#CF7A2A"      # цвета серий на графиках
NAVY, GREY = "#1E2761", "#9AA3B5"
IN_COLAB = "google.colab" in sys.modules
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| Colab" if IN_COLAB else "| локально")

Данные — условная фондовая площадка **«Меридиан»** (все компании вымышлены; ячейка работает ≈ 1 с). Запустите её.

In [ ]:
# Сквозной кейс курса «Анализ данных и машинное обучение на финансовых рынках» — «Меридиан»: условная
# фондовая площадка. Все компании, тикеры, котировки и новости вымышлены, совпадения с реальными компаниями
# случайны; генерация детерминирована (seed 2026), у всех получаются одинаковые числа. Только стандартная библиотека.
#   STOCKS      — 12 акций четырёх секторов: тикер, название, сектор, число акций (млн), лот
#   DATES       — торговые дни 09.01.2019–30.09.2026 (без выходных и праздников)
#   PRICES      — дневные данные акций: date, ticker, open, high, low, close (без корректировок), adj_close
#                 (с учётом дивидендов и сплитов), volume (шт.)
#   INDEX       — индекс широкого рынка MRDX: date, open, high, low, close
#   FX          — условный курс доллара: date, usd
#   NEWS        — новости компаний (кроме KLSU и TVRA): date, ticker, headline, label (разметка аналитика: позитив/негатив/нейтрально)
#   DIVIDENDS, SPLITS — корпоративные действия (дата отсечки, размер дивиденда на акцию; коэффициент сплита)
#   RAW_PRICES  — «сырая» выгрузка котировок трёх акций с типичными дефектами (ключ — RAW_DEFECTS)
#   ticks(ticker, start, end) — ЛЕНИВЫЙ генератор сделок внутри дня (время, цена, количество, сторона)
import math as _m
import random as _random
from datetime import date as _date, timedelta as _td

_SEED = 2026
_TITLE = "Меридиан"
_INDEX = "MRDX"
_R = _random.Random(_SEED)
_D0, _D1 = _date(2019, 1, 9), _date(2026, 9, 30)

_STOCKS = [  # тикер, название, сектор, бета рынка, бета курса, σ собств., лаг рынка, цена, акций (млн), див. доходн.
    ("BRZB", "Бриз-Банк", "финансы", 1.20, -0.10, 0.012, 0.00, 250, 2400, 0.08),
    ("AQLF", "Аквилон Финанс", "финансы", 1.05, -0.05, 0.014, 0.00, 85, 3100, 0.06),
    ("PKRV", "Покров Страхование", "финансы", 0.85, 0.00, 0.011, 0.00, 410, 420, 0.07),
    ("GRZN", "Горизонт-Нефть", "энергетика", 0.95, 0.55, 0.011, 0.00, 640, 1500, 0.10),
    ("SRMG", "Сарматгаз", "энергетика", 0.85, 0.45, 0.013, 0.00, 175, 5200, 0.09),
    ("VLTE", "Вольта Энерго", "энергетика", 0.70, 0.20, 0.010, 0.00, 32, 22000, 0.07),
    ("KLSU", "Колос-Юг", "агро", 0.60, 0.15, 0.015, 0.35, 120, 900, 0.04),
    ("TVRA", "Таврида-Агро", "агро", 0.60, 0.15, 0.015, 0.35, 118, 760, 0.00),
    ("NVGR", "Ниваград", "агро", 0.55, 0.10, 0.016, 0.35, 56, 1300, 0.03),
    ("PXLS", "Пиксель Софт", "технологии", 1.35, -0.30, 0.017, 0.00, 2400, 120, 0.00),
    ("NSFT", "Ноосфера Телеком", "технологии", 1.10, -0.20, 0.013, 0.00, 310, 1700, 0.02),
    ("KBRM", "Кибериум", "технологии", 1.45, -0.35, 0.020, 0.00, 95, 2600, 0.00)]
_LAG_UNTIL = _date(2024, 6, 30)       # после этой даты запаздывание агроакций почти исчезает (дрейф)
_LAG_AFTER = 0.0
_SPLIT = ("PXLS", _date(2023, 3, 15), 10)
_STRESS = {_date(2020, 3, 9): -4.0, _date(2020, 3, 12): -3.0, _date(2020, 3, 16): -3.5}
_HOLI = {(1, k) for k in range(1, 9)} | {(2, 23), (3, 8), (5, 1), (5, 9), (6, 12), (11, 4)}

STOCKS = [{"ticker": s[0], "name": s[1], "sector": s[2], "shares_mln": s[8], "lot": 10 if s[7] < 1000 else 1}
          for s in _STOCKS]


def _t(r, nu):
    """Стандартизованное t-распределение (дисперсия 1): «толстые хвосты»."""
    z = r.gauss(0, 1)
    chi = 2 * r.gammavariate(nu / 2, 1)
    return z / _m.sqrt(chi / nu) / _m.sqrt(nu / (nu - 2))


DATES = []
_d = _D0
while _d <= _D1:
    if _d.weekday() < 5 and (_d.month, _d.day) not in _HOLI:
        DATES.append(_d)
    _d += _td(days=1)
_T = len(DATES)

# ---------- рынок: GJR-GARCH(1,1) с t-шумом, условный кризис весны 2020 ----------
_MU, _OM, _AL, _GA, _BE = 0.00035, 3.0e-6, 0.04, 0.06, 0.90
_rm, _sig = [], []
_s2, _eps = 0.0001, 0.0
for _d in DATES:
    _s2 = _OM + (_AL + _GA * (_eps < 0)) * _eps ** 2 + _BE * _s2 if _rm else _s2
    _z = _t(_R, 6)
    _z = _STRESS.get(_d, _z * (1.15 if _z < 0 else 0.85) + 0.115)  # падения резче роста: отрицательная асимметрия
    _eps = _m.sqrt(_s2) * _z
    _rm.append(_MU + _eps)
    _sig.append(_m.sqrt(_s2))
# ---------- курс и секторы ----------
_fx = [0.00015 - 0.30 * (_rm[i] - _MU) + max(0.003, 0.004 + 0.25 * (_sig[i] - 0.01)) * _t(_R, 5) for i in range(_T)]
_SECT = sorted({s[2] for s in _STOCKS})
_fs = {k: [0.009 * _m.sqrt(_sig[i] / 0.01) * _t(_R, 5) for i in range(_T)] for k in _SECT}

# ---------- новости: шаблоны заголовков ----------
_POS = ["«{n}» увеличивает чистую прибыль на {p} %", "«{n}» повышает прогноз выручки на год",
        "«{n}» объявляет о выкупе акций на {b} млрд руб.", "Рейтинговое агентство повышает рейтинг «{n}»",
        "«{n}» получает крупный контракт на {b} млрд руб.", "Выручка «{n}» растёт на {p} % — выше ожиданий",
        "Совет директоров «{n}» рекомендует рекордные дивиденды", "«{n}» выходит на новый рынок и расширяет производство",
        "Аналитики улучшают оценку «{n}» после сильного отчёта", "«{n}» сокращает долг и снижает расходы"]
_NEG = ["Чистая прибыль «{n}» снижается на {p} %", "«{n}» понижает прогноз выручки на год",
        "«{n}» отменяет выплату дивидендов", "Рейтинговое агентство понижает рейтинг «{n}»",
        "«{n}» теряет крупный контракт", "Выручка «{n}» падает на {p} % — хуже ожиданий",
        "Против «{n}» подан иск на {b} млрд руб.", "«{n}» сообщает об убытке и росте долга",
        "Аналитики ухудшают оценку «{n}» после слабого отчёта", "«{n}» приостанавливает производство из-за аварии"]
_NEU = ["«{n}» проведёт годовое собрание акционеров", "«{n}» опубликует отчётность на следующей неделе",
        "«{n}» назначает нового финансового директора", "«{n}» представит стратегию на дне инвестора",
        "«{n}» меняет аудитора", "«{n}» разместит облигации на {b} млрд руб."]
_MIX = [("Несмотря на снижение выручки, «{n}» увеличивает прибыль на {p} %", 1),
        ("Прибыль «{n}» растёт, но компания понижает прогноз на год", -1),
        ("«{n}» снижает расходы, однако теряет крупный контракт", -1),
        ("Убыток «{n}» сокращается, компания возвращается к росту выручки", 1)]
_LAB = {1: "позитив", -1: "негатив", 0: "нейтрально"}

# ---------- акции ----------
_RS = _random.Random(_SEED + 1)
_rets, _news_eff, NEWS = {}, {}, []
_spread = 0.0
for _s in _STOCKS:
    _tk, _nm, _sec, _b, _g, _sd, _lag = _s[:7]
    _a = 0.00012 if _sec == "энергетика" else (0.0 if _sec == "технологии" else 0.00006)
    _r, _carry, _eff, _h, _e = [], 0.0, {}, 1.0, 0.0
    for _i, _d in enumerate(DATES):
        _lg = (_lag if _d <= _LAG_UNTIL else _LAG_AFTER) if _lag else 0.0
        _h = 0.02 + 0.12 * _e ** 2 + 0.86 * _h                    # собственный GARCH(1,1) шума акции
        _scale = (0.7 + 0.3 * _sig[_i] / 0.01) * _m.sqrt(_h)
        _e = _t(_RS, 5)
        _x = (_a + _b * (_rm[_i] - _MU) + _lg * (_rm[_i - 1] - _MU if _i else 0.0) + _g * _fx[_i] + _fs[_sec][_i]
              + _sd * _scale * _e + _carry)
        _carry = 0.0
        if _tk not in ("KLSU", "TVRA") and _RS.random() < 0.045:                  # новость о компании
            _u = _RS.random()
            _sent = 1 if _u < 0.40 else (-1 if _u < 0.75 else 0)
            _mix = _sent != 0 and _RS.random() < 0.12
            if _mix:
                _tpl, _sent = _RS.choice([x for x in _MIX])
            else:
                _tpl = _RS.choice({1: _POS, -1: _NEG, 0: _NEU}[_sent])
            _jump = _sent * (0.02 + 0.015 * abs(_RS.gauss(0, 1)))
            if _sent and _RS.random() < 0.12:
                _jump *= -0.6                                          # рынок «не согласился» с новостью
            if not _sent:
                _jump = 0.004 * _RS.gauss(0, 1)
            _x += _jump
            _carry = 0.25 * _jump                                      # дрейф после новости на следующий день
            _lab = _sent if _RS.random() > 0.08 else _RS.choice([v for v in (1, -1, 0) if v != _sent])
            _head = _tpl.format(n=_nm, p=_RS.randint(5, 45), b=_RS.randint(2, 60))
            _head = _head[0].upper() + _head[1:]
            NEWS.append({"date": _d.isoformat(), "ticker": _tk, "headline": _head, "label": _LAB[_lab]})
            _eff[_i] = _jump
        _r.append(_x)
    if _tk == "TVRA":                     # коинтеграция с KLSU: спред возвращается к среднему
        _r, _sp, _RC = [], [], _random.Random(_SEED + 5)
        for _i in range(_T):
            _new = 0.96 * _spread + 0.013 * _RC.gauss(0, 1)
            _r.append(_rets["KLSU"][_i] + (_new - _spread))
            _spread = _new
    _rets[_tk], _news_eff[_tk] = _r, _eff

# ---------- инсайдерский след: аномальный объём накануне части позитивных новостей ----------
_RI = _random.Random(_SEED + 9)
_pos_news = [(k, n) for k, n in enumerate(NEWS) if n["label"] == "позитив"]
_INSIDE = {(n["ticker"], n["date"]) for k, n in _RI.sample(_pos_news, 8)}
_POS_IDX = {d.isoformat(): i for i, d in enumerate(DATES)}

# ---------- цены, объёмы, дивиденды, сплит ----------
_RP = _random.Random(_SEED + 2)
PRICES, DIVIDENDS, SPLITS, _BY = [], [], [], {}
for _k, _s in enumerate(_STOCKS):
    _tk, _sd, _p0, _shares, _dy = _s[0], _s[5], _s[7], _s[8], _s[9]
    _p, _rows, _ev = float(_p0), [], []
    _base = 4.0e6 / (_p0 / 100) ** 0.6
    _ins = {_POS_IDX[d] - 1 for t, d in _INSIDE if t == _tk}
    _newsd = set(_news_eff[_tk])
    for _i, _d in enumerate(DATES):
        _r = _rets[_tk][_i]
        _prev = _p
        _div = 0.0
        if _dy and _d.month == 7 and 8 + _k <= _d.day and not any(e[0] == "div" and e[1].year == _d.year for e in _ev):
            _div = round(_dy * _prev * (0.8 + 0.4 * _RP.random()), 2)
            _ev.append(("div", _d, _div))
            DIVIDENDS.append({"ticker": _tk, "ex_date": _d.isoformat(), "dividend": _div})
        _p = _prev * _m.exp(_r) - _div
        if _tk == _SPLIT[0] and _d == _SPLIT[1]:
            _p /= _SPLIT[2]
            _ev.append(("split", _d, _SPLIT[2]))
            SPLITS.append({"ticker": _tk, "date": _d.isoformat(), "ratio": _SPLIT[2]})
        _gap = 0.3 * _r + 0.002 * _RP.gauss(0, 1)
        _o = (_prev - _div) * _m.exp(_gap) / (_SPLIT[2] if (_tk == _SPLIT[0] and _d == _SPLIT[1]) else 1)
        _intra = _sd * (0.6 + 0.4 * _sig[_i] / 0.01) + 0.5 * _sig[_i]
        _hi = max(_o, _p) * _m.exp(abs(_RP.gauss(0, 1)) * _intra * 0.6)
        _lo = min(_o, _p) * _m.exp(-abs(_RP.gauss(0, 1)) * _intra * 0.6)
        _tot = _m.sqrt((_sig[_i]) ** 2 + (_sd * (0.6 + 0.4 * _sig[_i] / 0.01)) ** 2)
        _vol = (_base * (1 + 0.08 * (_d.year - 2019)) * (_sig[_i] / 0.01) ** 0.5
                * _m.exp(0.35 * _RP.gauss(0, 1) + 0.45 * min(abs(_r) / _tot, 5)))
        if _i in _newsd:
            _vol *= 2.5
        if _i in _ins:
            _vol *= 6.0
        if _tk == _SPLIT[0] and _d >= _SPLIT[1]:
            _vol *= _SPLIT[2]
        _nd = 2 if _p < 1000 else 1
        _rows.append([_d.isoformat(), _tk, round(_o, _nd), round(_hi, _nd), round(_lo, _nd), round(_p, _nd),
                      None, int(round(_vol, -1))])
    # скорректированная цена: назад от последней даты, множители дивидендов и сплитов
    _f, _evd = 1.0, {e[1].isoformat(): e for e in _ev}
    for _j in range(len(_rows) - 1, -1, -1):
        _rows[_j][6] = round(_rows[_j][5] * _f, 4)
        e = _evd.get(_rows[_j][0])
        if e is not None and _j:
            _f *= (1 - e[2] / _rows[_j - 1][5]) if e[0] == "div" else 1 / e[2]
    _keys = ("date", "ticker", "open", "high", "low", "close", "adj_close", "volume")
    _BY[_tk] = [dict(zip(_keys, r)) for r in _rows]
for _i in range(_T):
    for _s in _STOCKS:
        PRICES.append(_BY[_s[0]][_i])

# ---------- индекс и курс ----------
_RX = _random.Random(_SEED + 3)
INDEX, FX = [], []
_lv, _usd = 1000.0, 70.0
for _i, _d in enumerate(DATES):
    _o = _lv * _m.exp(0.3 * _rm[_i] + 0.001 * _RX.gauss(0, 1))
    _lv *= _m.exp(_rm[_i])
    _w = 0.6 * _sig[_i]
    INDEX.append({"date": _d.isoformat(), "open": round(_o, 2),
                  "high": round(max(_o, _lv) * _m.exp(abs(_RX.gauss(0, 1)) * _w), 2),
                  "low": round(min(_o, _lv) * _m.exp(-abs(_RX.gauss(0, 1)) * _w), 2), "close": round(_lv, 2)})
    _usd *= _m.exp(_fx[_i])
    FX.append({"date": _d.isoformat(), "usd": round(_usd, 4)})

# ---------- «сырая» выгрузка с дефектами (3 акции) ----------
_RR = _random.Random(_SEED + 7)
RAW_PRICES, RAW_DEFECTS = [], {"пропущенный день": 0, "полный дубль строки": 0, "дата дд.мм.гггг": 0,
                               "цена строкой с запятой": 0, "цена ×100 (ошибка запятой)": 0,
                               "нулевой объём": 0, "зависшая цена (повтор вчерашней)": 0, "пропуск цены": 0}
for _tk in ("BRZB", "KLSU", "PXLS"):
    _prev = None
    for _row in _BY[_tk]:
        _x = {k: _row[k] for k in ("date", "ticker", "open", "high", "low", "close", "volume")}
        _u = _RR.random()
        if _u < 0.012:
            RAW_DEFECTS["пропущенный день"] += 1
            continue
        elif _u < 0.032:
            _x["date"] = _date.fromisoformat(_x["date"]).strftime("%d.%m.%Y"); RAW_DEFECTS["дата дд.мм.гггг"] += 1
        elif _u < 0.047:
            _x["close"] = f"{_x['close']:.2f}".replace(".", ","); RAW_DEFECTS["цена строкой с запятой"] += 1
        elif _u < 0.050:
            _x["close"] = round(_x["close"] * 100, 2); RAW_DEFECTS["цена ×100 (ошибка запятой)"] += 1
        elif _u < 0.056:
            _x["volume"] = 0; RAW_DEFECTS["нулевой объём"] += 1
        elif _u < 0.061 and _prev is not None and isinstance(_prev["close"], float):
            _x["close"] = _prev["close"]
            RAW_DEFECTS["зависшая цена (повтор вчерашней)"] += 1
        elif _u < 0.068:
            _x["close"] = None; RAW_DEFECTS["пропуск цены"] += 1
        RAW_PRICES.append(_x)
        if _RR.random() < 0.01:
            RAW_PRICES.append(dict(_x)); RAW_DEFECTS["полный дубль строки"] += 1
        _prev = _x


def ticks(ticker: str, start: str, end: str, seed: int = _SEED):
    """ЛЕНИВЫЙ генератор сделок за торговые дни [start, end]: кортежи (время «ГГГГ-ММ-ДД ЧЧ:ММ:СС», тикер, цена,
    количество, сторона «B»/«S»). Цены открытия и закрытия дня совпадают с PRICES; около 2–6 тыс. сделок в день."""
    for row in _BY[ticker]:
        if not (start <= row["date"] <= end):
            continue
        r = _random.Random(f"{seed}-{ticker}-{row['date']}")
        n = max(300, min(8000, row["volume"] // 900))
        u = sorted(r.betavariate(0.65, 0.65) for _ in range(n))          # U-образная активность в течение сессии
        lo, lc = _m.log(row["open"]), _m.log(row["close"])
        sig = max(_m.log(row["high"] / row["low"]) / 1.6, 0.002)          # волатильность дня — по диапазону H/L
        w, tp, path = 0.0, 0.0, []
        for x in u:
            w += r.gauss(0, 1) * _m.sqrt(max(x - tp, 1e-9)) * sig
            tp = x
            path.append(w)
        wl = path[-1]
        half = max(0.01, row["close"] * 0.0004)
        for x, wx in zip(u, path):
            side = "B" if r.random() < 0.5 else "S"
            mid = _m.exp(lo + wx - x * wl + x * (lc - lo))
            sec = 36000 + int(x * 31200)                                   # 10:00–18:40
            yield (f"{row['date']} {sec // 3600:02d}:{sec // 60 % 60:02d}:{sec % 60:02d}", ticker,
                   round(mid + (half if side == "B" else -half), 2), int(r.paretovariate(1.6) * 10), side)

print(f"{_TITLE}: акций {len(STOCKS)}, торговых дней {len(DATES)} ({DATES[0]} – {DATES[-1]}), строк котировок {len(PRICES)}, новостей {len(NEWS)}")

## 1. Данные кейса 📘

Справочник акций, котировки «длинной» таблицей и таблица «дата × тикер».

In [ ]:
# Данные кейса: справочник акций, «длинная» таблица котировок -> таблица «дата × тикер»
st = pd.DataFrame(STOCKS)
px = pd.DataFrame(PRICES)
px["date"] = pd.to_datetime(px.date)
w = px.pivot(index="date", columns="ticker", values="adj_close")[st.ticker]      # скорректированные цены
idx = pd.DataFrame(INDEX).assign(date=lambda d: pd.to_datetime(d.date)).set_index("date").close
print(st.groupby("sector").ticker.apply(list).to_dict())
print(pd.DataFrame(NEWS).label.value_counts().to_dict())
w.tail(3).round(2)

## 2. Очистка сырой выгрузки 📘

Даты двух форматов, цены строкой, дубли, ошибка ×100, зависшие цены, календарь торгов. Сравните журнал с ключом генератора.

In [ ]:
# Сырая выгрузка и конвейер очистки (как в конспекте, раздел 1.2)
raw = pd.DataFrame(RAW_PRICES)

def clean_prices(raw, calendar):
    d, log = raw.copy(), {}
    iso = pd.to_datetime(d.date, format="%Y-%m-%d", errors="coerce")
    d["date"] = iso.fillna(pd.to_datetime(d.date, format="%d.%m.%Y", errors="coerce"))
    log["дата дд.мм.гггг"] = int(iso.isna().sum())
    d["close"] = pd.to_numeric(d.close.map(lambda v: v.replace(",", ".") if isinstance(v, str) else v))
    n0 = len(d)
    d = d.drop_duplicates().sort_values(["ticker", "date"])
    log["дубли"] = n0 - len(d)
    med = d.groupby("ticker").close.transform(lambda s: s.rolling(11, center=True, min_periods=3).median())
    big = d.close > 20 * med
    d.loc[big, "close"] /= 100
    log["цена ×100"] = int(big.sum())
    stale = (d.close == d.groupby("ticker").close.shift(1)) | (d.close > d.high) | (d.close < d.low)
    d.loc[stale, "close"] = np.nan
    log["зависшая / несогласованная"] = int(stale.sum())
    full = pd.MultiIndex.from_product([sorted(d.ticker.unique()), pd.to_datetime(calendar)], names=["ticker", "date"])
    d = d.set_index(["ticker", "date"])
    log["пропущенные дни"] = len(full) - len(d)
    return d.reindex(full), log

clean, log = clean_prices(raw, DATES)
print(log)
print("ключ генератора:", RAW_DEFECTS)

## 3. Корпоративные действия 📘

Сплит превращает цену закрытия в «обвал», дивиденды — в гэпы; доходности считаем по `adj_close`.

In [ ]:
# Корпоративные действия: сплит PXLS и дивиденды GRZN
g = px[px.ticker == "PXLS"].set_index("date")
print(pd.DataFrame({"close": g.close, "adj_close": g.adj_close,
                    "r по close": np.log(g.close).diff().round(4)}).loc["2023-03-13":"2023-03-16"])
dv = pd.DataFrame(DIVIDENDS)
print(dv.groupby("ticker").dividend.count().to_dict())
fig, ax = plt.subplots(figsize=(9, 3))
gz = px[px.ticker == "GRZN"].set_index("date")
ax.plot(gz.close, color=ORANGE, label="close"); ax.plot(gz.adj_close, color=BLUE, ls="--", label="adj_close")
ax.set_title("GRZN: цена и скорректированная цена"); ax.set_ylabel("₽"); ax.legend(); plt.show()

## 4. Доходности, риск и просадка 📘

Лог-доходности, годовые доходность и волатильность, Шарп, просадка.

In [ ]:
# Доходности и таблица «риск — доходность»
r = np.log(w).diff().dropna()
ri = np.log(idx).diff().dropna()

def risk_table(r):
    t = pd.DataFrame({"доходность, % год.": r.mean() * 252 * 100, "волатильность, % год.": r.std() * np.sqrt(252) * 100})
    t["Шарп"] = t.iloc[:, 0] / t.iloc[:, 1]
    wealth = np.exp(r.cumsum())
    t["макс. просадка, %"] = (wealth / wealth.cummax() - 1).min() * 100
    return t.round(2)

risk_table(r.assign(MRDX=ri))

## 5. Стилизованные факты 📘

Эксцесс, Жарке–Бера, ACF доходностей и их модулей.

In [ ]:
# Стилизованные факты: толстые хвосты и кластеризация волатильности
def acf(x, nlags=10):
    x = np.asarray(x, float) - np.mean(x)
    return np.array([x[k:] @ x[:-k] / (x @ x) for k in range(1, nlags + 1)])

def ljung_box(x, m=10):
    n, rho = len(x), acf(x, m)
    q = n * (n + 2) * np.sum(rho ** 2 / (n - np.arange(1, m + 1)))
    return q, stats.chi2.sf(q, m)

print(f"эксцесс {stats.kurtosis(ri):.2f}, Жарке–Бера p = {stats.jarque_bera(ri).pvalue:.2g}")
print("ACF r:", acf(ri, 5).round(3), "| ACF |r|:", acf(ri.abs(), 5).round(3))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(ri * 100, bins=80, color=BLUE, density=True)
xs = np.linspace(-8, 8, 300)
ax[0].plot(xs, stats.norm.pdf(xs, ri.mean() * 100, ri.std() * 100), color=ORANGE, label="нормальное")
ax[0].set_xlabel("доходность MRDX, %"); ax[0].legend()
ax[1].bar(range(1, 31), acf(ri.abs(), 30), color=ORANGE); ax[1].set_title("ACF модулей доходности")
plt.tight_layout(); plt.show()

## 6. VaR и бэктест 📘

Прогноз на завтра — только по прошлым данным (`.shift(1)`); пробои и тест Купика.

In [ ]:
# VaR 99 % на завтра: историческое окно, нормальное, EWMA; бэктест и тест Купика
x = r.BRZB
q = stats.norm.ppf(0.99)
var = {"историческое": -x.rolling(250).quantile(0.01).shift(1),
       "нормальное": -(x.rolling(250).mean() - q * x.rolling(250).std()).shift(1),
       "EWMA": q * np.sqrt((x ** 2).ewm(alpha=0.06, adjust=False).mean()).shift(1)}

def kupiec(k, n, p=0.01):
    ph = k / n
    ll0 = (n - k) * np.log(1 - p) + k * np.log(p)
    ll1 = (n - k) * np.log(1 - ph) + (k * np.log(ph) if k else 0.0)
    return stats.chi2.sf(-2 * (ll0 - ll1), 1)

for name, v in var.items():
    m = v.notna() & (x.index >= "2020-01-01")
    k = int((x[m] < -v[m]).sum())
    print(f"{name:<13} пробоев {k} из {m.sum()} ({k / m.sum():.2%}), p Купика = {kupiec(k, int(m.sum())):.3f}")

## 7. Бета, корреляции, портфель 📘

Бета к индексу, тепловая карта корреляций, портфель минимальной дисперсии.

In [ ]:
# Бета, корреляции, портфель минимальной дисперсии
beta = r.apply(lambda s: s.cov(ri) / ri.var())
print(beta.round(2).to_dict())
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(r.corr(), cmap="Blues", vmin=0, vmax=0.7)
ax.set_xticks(range(12), r.columns, rotation=60); ax.set_yticks(range(12), r.columns)
plt.colorbar(im); ax.set_title("корреляции доходностей"); plt.show()
train, test = r.loc[:"2023"], r.loc["2024":]
w_mv = np.linalg.solve(train.cov().to_numpy(), np.ones(12)); w_mv /= w_mv.sum()
port = lambda wt, rets: np.log1p(np.expm1(rets) @ wt)
for name, wt in [("равные веса", np.full(12, 1 / 12)), ("мин. дисперсия", w_mv)]:
    print(f"{name:<15} волатильность 2024–2026: {port(wt, test).std() * np.sqrt(252):.1%}")

---

# Задания ✏️

Пишите код вместо `...` / `# ваш код`, затем запускайте **ячейку проверки** под заданием.

## Задание 1. Разбор дат и чисел

Два формата дат, десятичная запятая, пропуски.

In [ ]:
# Задание 1. Две функции разбора «сырых» полей: to_date(s) — Series строк в двух форматах («2024-01-15» и
# «15.01.2024») -> datetime64; to_number(s) — Series из float, строк с запятой («247,93») и None -> float.
# Примените к raw: dates, closes; n_ddmm — сколько дат было в формате дд.мм.гггг; n_nan — сколько NaN в closes.
def to_date(s):
    ...

def to_number(s):
    ...

In [ ]:
assert to_number(pd.Series(["1,5", None, 2.0])).fillna(-1).tolist() == [1.5, -1, 2.0]
assert to_date(pd.Series(["2024-01-15", "15.01.2024"])).nunique() == 1
assert dates.notna().all() and n_ddmm == 105 and n_nan == 34
print("Задание 1: верно")

## Задание 2. Корректировка на дивиденды

Множители назад от последней даты.

In [ ]:
# Задание 2. Восстановите скорректированную цену GRZN сами: идём от последней даты назад, множитель f = 1;
# в каждый день отсечки (DIVIDENDS) для всех более ранних цен f умножается на (1 − D / close накануне).
# Результат — Series my_adj (индекс — даты); err — максимальная относительная ошибка относительно adj_close.
# ваш код

In [ ]:
assert len(my_adj) == len(gz) and err < 1e-3
assert my_adj.iloc[-1] == gz.close.iloc[-1] and my_adj.iloc[0] < gz.close.iloc[0] * 0.7
print("Задание 2: верно")

## Задание 3. Годовая статистика индекса

Группировка по годам.

In [ ]:
# Задание 3. Годовая статистика индекса: таблица yearly (индекс — год) с колонками "доходность" (простая за год:
# exp(сумма лог-доходностей) − 1) и "волатильность" (годовая: std · √252). best_year, worst_year — годы с наибольшей
# и наименьшей доходностью. (2019 и 2026 — неполные годы, это нормально.)
# ваш код

In [ ]:
assert list(yearly.columns) == ["доходность", "волатильность"] and len(yearly) == 8
assert yearly.доходность.round(4).to_dict() == {2019: 0.0952, 2020: 0.2371, 2021: 0.1032, 2022: 0.2086, 2023: -0.0394, 2024: 0.578, 2025: 0.1553, 2026: -0.0087}
assert best_year == 2024 and worst_year == 2023
print("Задание 3: верно")

## Задание 4. Толстые хвосты

Эксцесс, Жарке–Бера, «4 сигмы».

In [ ]:
# Задание 4. Толстые хвосты у всех 12 акций: таблица tails (индекс — тикер) с колонками "эксцесс" (stats.kurtosis),
# "JB p", "дней |z| > 4" (z — стандартизованная доходность) и "ожидалось при N" (n · 2 · P(Z > 4)).
# У скольких акций критерий Жарке–Бера отвергает нормальность при α = 0,01 (n_reject)?
# ваш код

In [ ]:
assert list(tails.columns) == ["эксцесс", "JB p", "дней |z| > 4", "ожидалось при N"]
assert tails.эксцесс.astype(float).round(2).to_dict() == {'BRZB': 1.93, 'AQLF': 7.33, 'PKRV': 1.97, 'GRZN': 7.81, 'SRMG': 2.57, 'VLTE': 2.69, 'KLSU': 2.94, 'TVRA': 1.76, 'NVGR': 2.04, 'PXLS': 2.8, 'NSFT': 3.59, 'KBRM': 2.42}
assert n_reject == 12 and (tails["дней |z| > 4"] > tails["ожидалось при N"]).all()
print("Задание 4: верно")

## Задание 5. Кластеризация волатильности

ACF и Льюнг–Бокс по 12 акциям.

In [ ]:
# Задание 5. Кластеризация волатильности по акциям: таблица clus (индекс — тикер): "ACF1 r", "ACF1 |r|",
# "LB p r", "LB p |r|" (Льюнг–Бокс, 10 лагов, функции из разбора). share_abs — доля акций, у которых Льюнг–Бокс для
# |r| отвергает H₀ при α = 0,05; mean_ratio — во сколько раз средний ACF1 |r| больше среднего модуля ACF1 r.
# ваш код

In [ ]:
assert clus.shape == (12, 4) and share_abs == 1.0
assert mean_ratio > 2
print("Задание 5: верно")

## Задание 6. VaR и ES портфеля

Исторические меры и бэктест EWMA-прогноза.

In [ ]:
# Задание 6. Риск равновзвешенного портфеля 12 акций (ежедневная ребалансировка; port из разбора).
# ew — его дневные лог-доходности за весь период. Для 2024–2026 гг.: исторические var99 и es99 (положительные числа).
# Прогноз EWMA-VaR 99 % на завтра (как в разборе) для ew: число пробоев n_br и доля br_rate с 2020 г.,
# p-значение Купика p_kup.
# ваш код

In [ ]:
assert abs(var99 - 0.02723) < 1e-4 and abs(es99 - 0.03375) < 1e-4 and es99 > var99
assert 0 <= p_kup <= 1 and 0.005 < br_rate < 0.04
print("Задание 6: верно")

## Задание 7. Стабильность беты

Беты по периодам и секторам.

In [ ]:
# Задание 7. Стабильность беты. betas — таблица (индекс — тикер): "β 2019–2023", "β 2024–2026" (cov / var по
# соответствующему периоду), "сектор". sector_beta — средние беты по секторам за 2019–2023 (Series, индекс — сектор).
# Какой сектор самый «агрессивный» (aggressive)?
# ваш код

In [ ]:
assert betas["β 2019–2023"].round(3).to_dict() == {'BRZB': 1.249, 'AQLF': 0.99, 'PKRV': 0.848, 'GRZN': 0.744, 'SRMG': 0.797, 'VLTE': 0.657, 'KLSU': 0.577, 'TVRA': 0.617, 'NVGR': 0.512, 'PXLS': 1.436, 'NSFT': 1.202, 'KBRM': 1.572}
assert aggressive == "технологии" and sector_beta.idxmin() == "агро"
print("Задание 7: верно")

## Задание 8 ★. Мини-исследование: портфели на новых данных

Четыре способа задать веса.

In [ ]:
# Задание 8 ★ (мини-исследование). Сравните на 2024–2026 гг. четыре портфеля, веса которых оценены по 2019–2023 гг.:
# "равные веса", "обратная волатильность" (wᵢ ∝ 1/σᵢ), "мин. дисперсия" (w = Σ⁻¹1 / 1ᵀΣ⁻¹1) и
# "мин. дисперсия без коротких" (отрицательные веса обнулить, остальные нормировать к 1).
# Таблица cmp (индекс — портфель): "вол-ть обуч., %", "вол-ть тест, %", "доходность тест, % год.", "Шарп тест",
# "просадка тест, %". Вывод 5–7 предложений: что дала оптимизация, почему на тесте преимущество меньше.
# ваш код

In [ ]:
assert cmp.shape == (4, 5)
assert cmp["вол-ть обуч., %"].idxmin() == "мин. дисперсия"
assert cmp.loc["мин. дисперсия", "вол-ть тест, %"] < cmp.loc["равные веса", "вол-ть тест, %"]
print("Задание 8: таблица построена; вывод проверит преподаватель")

---

## Что вы теперь умеете

| Задача | Инструмент |
|---|---|
| очистка котировок | `pd.to_datetime(format=…)`, `drop_duplicates`, медиана соседей, `reindex` |
| корпоративные действия | `adj_close`, множители 1/k и 1 − D/P |
| доходности и риск | `np.log(w).diff()`, √252, Шарп, просадка |
| стилизованные факты | `stats.kurtosis`, `jarque_bera`, `acf`, Льюнг–Бокс |
| VaR и ES | `rolling(…).quantile`, EWMA, `.shift(1)`, тест Купика |
| бета и портфели | `cov / var`, `np.linalg.solve(S, 1)` |

**Дальше — тема 2 «Методы машинного обучения в анализе финансовых рынков».** Факторные
модели, классификация направления, деревья и ансамбли, кластеризация акций, временные ряды и аномалии.